In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load both files
base_df = pd.read_csv('../data/raw/storm_lineups_base.csv')
adv_df = pd.read_csv('../data/raw/storm_lineups_advanced.csv')

# Check the advanced columns we care about
key_cols = ['SEASON', 'GROUP_NAME', 'MIN', 'GP', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'PACE', 'POSS']
print("Do these columns exist in advanced?")
print([col for col in key_cols if col in adv_df.columns])
print("\nMissing:")
print([col for col in key_cols if col not in adv_df.columns])

Do these columns exist in advanced?
['SEASON', 'GROUP_NAME', 'MIN', 'GP', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'PACE', 'POSS']

Missing:
[]


In [2]:
# Focus on just what we need
df = adv_df[['SEASON', 'GROUP_NAME', 'MIN', 'GP', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'PACE', 'POSS']].copy()

# How many lineups per season?
print("Lineups per season:")
print(df.groupby('SEASON')['GROUP_NAME'].count().to_string())

# Basic stats on ratings
print("\nRating ranges:")
print(df[['OFF_RATING', 'DEF_RATING', 'NET_RATING']].describe())

# How many lineups have meaningful possession counts?
print("\nPossession distribution:")
print(df['POSS'].describe())
print(f"\nLineups with 50+ possessions: {len(df[df['POSS'] >= 50])}")
print(f"Lineups with 100+ possessions: {len(df[df['POSS'] >= 100])}")
print(f"Lineups with 20+ possessions: {len(df[df['POSS'] >= 20])}")

Lineups per season:
SEASON
2000    295
2001    190
2002    202
2003    178
2004    128
2005    144
2006    214
2007    164
2008    217
2009    157
2010    122
2011    110
2012    167
2013    118
2014    188
2015    179
2016    138
2017    129
2018    150
2019    132
2020    125
2021    159
2022    148
2023    212
2024    134

Rating ranges:
        OFF_RATING   DEF_RATING   NET_RATING
count  4100.000000  4100.000000  4100.000000
mean     79.981610    91.342902   -11.361415
std      55.588054    58.492141    82.326540
min       0.000000     0.000000  -300.000000
25%      44.400000    60.000000   -50.000000
50%      83.300000    91.700000     0.000000
75%     108.700000   117.600000    30.000000
max     300.000000   300.000000   300.000000

Possession distribution:
count    4100.000000
mean       16.510976
std        52.722023
min         0.000000
25%         3.000000
50%         6.000000
75%        13.000000
max      1094.000000
Name: POSS, dtype: float64

Lineups with 50+ possessions: 

In [3]:
# Lineups per season above our threshold
threshold = 20
df_filtered = df[df['POSS'] >= threshold].copy()

print("Lineups per season (20+ possessions):")
season_counts = df_filtered.groupby('SEASON')['GROUP_NAME'].count()
print(season_counts.to_string())

# Flag any seasons with suspiciously few lineups
print(f"\nSeasons with fewer than 5 lineups: {list(season_counts[season_counts < 5].index)}")
print(f"Total lineups for analysis: {len(df_filtered)}")

Lineups per season (20+ possessions):
SEASON
2000    29
2001    27
2002    35
2003    26
2004    23
2005    30
2006    29
2007    31
2008    31
2009    26
2010    25
2011    23
2012    41
2013    29
2014    28
2015    29
2016    27
2017    30
2018    25
2019    30
2020    20
2021    28
2022    27
2023    41
2024    30

Seasons with fewer than 5 lineups: []
Total lineups for analysis: 720


In [4]:
# Season-level aggregation — weighted by possessions
season_summary = df_filtered.groupby('SEASON').apply(
    lambda x: pd.Series({
        'OFF_RATING': (x['OFF_RATING'] * x['POSS']).sum() / x['POSS'].sum(),
        'DEF_RATING': (x['DEF_RATING'] * x['POSS']).sum() / x['POSS'].sum(),
        'NET_RATING': (x['NET_RATING'] * x['POSS']).sum() / x['POSS'].sum(),
        'TOTAL_POSS': x['POSS'].sum(),
        'LINEUP_COUNT': len(x)
    })
).reset_index()

print(season_summary)

    SEASON  OFF_RATING  DEF_RATING  NET_RATING  TOTAL_POSS  LINEUP_COUNT
0     2000   78.014902   89.954676  -11.936896       973.0          29.0
1     2001   84.038990   86.079731   -2.035152      1485.0          27.0
2     2002   95.643380   88.529319    7.138840      1586.0          35.0
3     2003  100.527205   91.628646    8.919251      1735.0          26.0
4     2004   98.347577   87.371965   10.977124      1919.0          23.0
5     2005   96.420795   92.038958    4.374458      1938.0          30.0
6     2006  100.653517   93.765592    6.863951      1706.0          29.0
7     2007  102.957227   91.138938   11.795280      2034.0          31.0
8     2008   95.437319   86.340323    9.084260      1798.0          31.0
9     2009   95.653025   90.724811    4.924953      2116.0          26.0
10    2010  102.332363   92.050134   10.280215      2234.0          25.0
11    2011   94.511962   88.932397    5.591754      2207.0          23.0
12    2012   91.866482   90.224322    1.628392     